In [ ]:
!pip install -q --upgrade langchain langchain-community langchain-core chromadb sentence-transformers pypdf python-docx pandas groq openpyxl google-generativeai

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 3.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 3.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 136.9/136.9 kB 5.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 41.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 558.3/558.3 kB 32.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 69.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 349.5/349.5 kB 23.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 253.0/253.0 kB 17.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.9/10.9 MB 99.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.7/143.7 kB 10.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 15.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 91.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [ ]:
!pip install -q --upgrade google-generativeai

In [ ]:
import os
from dotenv import load_dotenv
load_dotenv()  # Load environment variables from a local .env file if it exists

# Use GROQ_API_KEY from environment variables (local .env or system env)
if "GROQ_API_KEY" not in os.environ:
    try:
        from google.colab import userdata
        os.environ["GROQ_API_KEY"] = userdata.get("GROQ_API_KEY")
    except ImportError:
        print("GROQ_API_KEY not found in environment variables.")

In [ ]:
!pip install python-docx

In [ ]:
!pip install pypdf

In [ ]:
from google.colab import files
import pandas as pd
from docx import Document
from pypdf import PdfReader # Changed from langchain_community.document_loaders.PyPDFLoader

uploaded = files.upload()

documents = []
processed_files = set() # To keep track of processed file names

for file_name in uploaded.keys():

    if file_name in processed_files:
        print(f"Skipping duplicate file: {file_name}")
        continue

    print(f"Processing: {file_name}")
    processed_files.add(file_name)

    text_content = ""

    # ---------- PDF ----------
    if file_name.lower().endswith(".pdf"):
        reader = PdfReader(file_name)
        pages = []
        for page in reader.pages:
            txt = page.extract_text()
            if txt:
                pages.append(txt)
        text_content = "\n".join(pages)

    # ---------- Word ----------
    elif file_name.lower().endswith(".docx"):
        doc = Document(file_name)
        text_content = "\n".join([para.text for para in doc.paragraphs])

    # ---------- CSV ----------
    elif file_name.lower().endswith(".csv"):
        df = pd.read_csv(file_name)
        text_content = df.to_string(index=False)

    # ---------- Excel ----------
    elif file_name.lower().endswith((".xlsx", ".xls")):
        excel = pd.ExcelFile(file_name)

        sheet_text = []

        for sheet in excel.sheet_names:
            df = pd.read_excel(file_name, sheet_name=sheet)

            sheet_text.append(
                f"\nSheet: {sheet}\n{df.to_string(index=False)}"
            )

        text_content = "\n".join(sheet_text)

    # ---------- TXT ----------
    elif file_name.lower().endswith(".txt"):
        with open(file_name, "r", encoding="utf-8") as f:
            text_content = f.read()

    else:
        print(f"Unsupported file: {file_name}")
        continue

    # Store as a normal Python dictionary
    documents.append({
        "text": text_content,
        "source": file_name
    })

print(f"\nSuccessfully loaded {len(documents)} documents.")

Saving GOWTHAM_RESUME_MULTICORE.pdf to GOWTHAM_RESUME_MULTICORE.pdf
Processing: GOWTHAM_RESUME_MULTICORE.pdf

Successfully loaded 1 documents.


In [ ]:
import sys
!{sys.executable} -m pip install langchain-community

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.documents import Document # Import Document class if needed for consistency or future use

splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50
)

# Extract text content from the 'documents' list of dictionaries
# and create LangChain Document objects (or just a list of strings if preferred by splitter)
# For compatibility with create_documents, we'll pass the raw text for now.

# Assuming 'documents' is a list of dictionaries like [{'text': '...', 'source': '...'}]
# We need to extract the 'text' part from each.

texts_to_split = [doc['text'] for doc in documents]

docs = splitter.create_documents(texts_to_split)

In [ ]:
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma
import os

# Ensure directory exists
if not os.path.exists('./chroma_db'):
    os.makedirs('./chroma_db')

embedding = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

# Initialize Chroma with persist_directory
vectorstore = Chroma.from_documents(
    docs,
    embedding,
    persist_directory="./chroma_db"
)
retriever = vectorstore.as_retriever()
print("Vector store initialized and persisted to ./chroma_db ✅")

/tmp/ipykernel_622/452743777.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.embeddings import HuggingFaceEmbeddings
/tmp/ipykernel_622/452743777.py:9: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embedding = HuggingFaceEmbeddings(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Vector store initialized and persisted to ./chroma_db ✅


In [ ]:
from groq import Groq
import os

client = Groq(api_key=os.environ["GROQ_API_KEY"])

def ask_llm(question, context):
    prompt = f"""
    Based ONLY on the following context, answer the question below.
    If the answer cannot be found in the context, please state that clearly.

    Context:
    {context}

    Question: {question}
    Answer:"""

    completion = client.chat.completions.create(
        model="llama-3.3-70b-versatile",
        messages=[{"role": "user", "content": prompt}],
        temperature=0.7
    )

    return completion.choices[0].message.content

In [ ]:
query = input("What would you like to ask about the document? ")
relevant_docs = retriever.invoke(query)
context = "\n\n".join([doc.page_content for doc in relevant_docs])

answer = ask_llm(query, context)
print(answer)

What would you like to ask about the document? EDUCATION QUALIFICATION
Based on the provided context, the education qualification is:

- B.E – Computer Science and Engineering (2023 – Present) at Sri Krishna College of Engineering and Technology, Coimbatore, with a CGPA of 8.26 / 10.0
- Higher Secondary Education (2021 – 2023) at T.P.P Government Higher Secondary School, Namakkal.

Note: The context does not mention any degree completion, as the B.E course is mentioned as "2023 – Present", indicating that it is ongoing.
